# Classification, overfitting, and neural networks

**MedLIME Learning Track — Notebook 03/12** · Curriculum weeks 9–16 · Suggested time: 60–100 minutes

**Before starting:** Notebooks 01–02; basic Python arrays.

**Learning goal:** Train a low-compute classifier and separate predictive accuracy from explanation quality.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## From numbers to predictions
A classifier converts image features into a score. We will use a separate train and test set and compare logistic regression with a tiny neural network. We extract the maximum brightness in each 8×8 region; **this engineered feature is part of the teaching setup**, not a clinical biomarker.

**Mini-math:** logistic regression uses \(p(y=1|x)=1/(1+e^{-(w^Tx+b)})\). A loss function penalizes wrong probabilities.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
Xtrain,ytrain,_=dataset(220,seed=100)
Xtest,ytest,_=dataset(100,seed=200)
def features(images):
    a=np.asarray(images)
    return a.reshape(len(a),4,8,4,8).max(axis=(2,4)).reshape(len(a),16)
clf=LogisticRegression(max_iter=600).fit(features(Xtrain),ytrain)
probs=clf.predict_proba(features(Xtest))[:,1]
print('Holdout accuracy:',accuracy_score(ytest,probs>=.5))
print('Confusion matrix:\n',confusion_matrix(ytest,probs>=.5))

## A two-layer neural network
Here PyTorch optimizes weights using gradients. In later notebooks we use a fixed deterministic black box instead, because stable examples keep attention on the explanation methods; you may substitute your own trained classifier when ready.

In [ ]:
import torch
from torch import nn
torch.manual_seed(4)
net=nn.Sequential(nn.Linear(16,24),nn.ReLU(),nn.Linear(24,1))
optimizer=torch.optim.Adam(net.parameters(),lr=.04)
train_x=torch.tensor(features(Xtrain),dtype=torch.float32)
train_y=torch.tensor(ytrain[:,None],dtype=torch.float32)
for epoch in range(45):
    logits=net(train_x)
    loss=nn.functional.binary_cross_entropy_with_logits(logits,train_y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
with torch.no_grad():
    test_scores=torch.sigmoid(net(torch.tensor(features(Xtest),dtype=torch.float32))).numpy().ravel()
print('Tiny MLP holdout accuracy:',accuracy_score(ytest,test_scores>=.5))
print('Last training BCE loss:',round(float(loss.detach()),4))

## Is a correct classification enough?
Visualize a correct image, label, and probability. A probability is **not** a saliency map; it contains no location information. Also note that evaluating on the same samples used for training would conceal overfitting.

In [ ]:
idx=int(np.flatnonzero(ytest==1)[0])
show_images([Xtest[idx],(Xtest[idx]>.7).astype(float)],
            [f'positive image; p={probs[idx]:.2f}','crude threshold, NOT explanation'])
plt.show()

## What you should now understand
- Separate training and test sets before evaluating a model.
- Predictive accuracy does not imply correct localization.
- Neural network parameters are learned by minimizing a loss using gradients.

## Try it yourself (do not just read the code)
1. Reduce the training set to 20 images. What happens to holdout accuracy?
2. Compare the engineered max-intensity features with patch-average intensity.

## Exit ticket
**Question:** Why might the 16 engineered features fail on a real mammogram or chest X-ray?

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.